In [ ]:

#   %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%% Headers, Imports, and Functions %%%%%%%%%%%%%%%%%%%%%%%%%%

import datetime
from datetime import datetime as dt
from datetime import date 
from datetime import timedelta
import time
import re
from pandas import read_csv
import pandas as pd
import json
import sys
import random
import numpy as np

from dateutil import parser
from dateutil.relativedelta import relativedelta
import ast

def maximize_screen():
    from IPython.display import display,HTML
    display(HTML("<style>:root{--jp-notebook-max-width:100%!important;}</style>"))

def  timestamp():
    today=str(datetime.date.today()).split('-')
    year=today[0]
    month=today[1]
    day=today[2]
    a=str(datetime.datetime.now())
    time_split=re.split(':|',a)
    hour=time_split[1]
    timestamp=year+month+day+hour
    return timestamp

def  display_dataframe_segments(dataframe_df):
    for i in range(0,len(dataframe_df),50):
        print(dataframe_df.iloc[i:i+50])
        input('')

def clean_dataframe(dataframe_df,column):
    delete_list=['Pvt ','PTE ','Dr. ','Rev ','MAJ ','GEN ','Mrs ','CPT ',',PFC ','SGT ','Col ','LT ','Dr ',\
                ' Sr',' Jr','\.','Minor ','Infant ','Boy ','Girl ','Twins ','PFC','TEC5'\
                'PVT','S Sgt','Capt','Pfc','TSgt','COL','Tec5','TEC4','1','2','III','II','Lt','CPL','SSgt','Sgt',\
                'Staff Sergeant','TEC 5','Maj','GM3','TEC 4','Tec4','RM3','Corp','T 5','MS','  ']
    dataframe_df=dataframe_df.copy()
    for item in delete_list:
        dataframe_df[column]= dataframe_df[column].astype(str).replace(item, ' ', regex=True).str.strip()
    return dataframe_df      

def discard_empty_sets(dataframe_df,column):
    dataframe_df[column]=dataframe_df[column].apply(
            lambda x:x.discard('') or x if isinstance(x,set) else x)
    return dataframe_df

def  trim_dataframe_set(dataframe_df,column):
    dataframe_df[column]=dataframe_df[column].apply(
            lambda x:{item.strip() for item in x if isinstance(item,str)} if isinstance(x,set) else x)
    return dataframe_df

def match_sets(soldier_match_list_df,search,column_search): 
    search_set= ast.literal_eval(search)
    if isinstance(soldier_match_list_df[column_search].iloc[0], set)==False:
        soldier_match_list_df.loc[:,column_search] = soldier_match_list_df[column_search].apply(
            lambda x: ast.literal_eval(x) if isinstance(x, str) and x.startswith('{') else set()) 
    soldier_match_list_df.loc[:,column_search] = soldier_match_list_df[column_search].apply(
        lambda x: {'nan'} if pd.isna(x) else x)
    soldier_match_list_df = soldier_match_list_df.copy()
    soldier_match_list_df.loc[:,'set_difference']=search_set-soldier_match_list_df[column_search]
    return soldier_match_list_df

def discard_set_one_character(data_set_df,column):
    data_set_df=data_set_df.copy()
    data_set_df[column] = data_set_df[column].apply(
        lambda x: {item for item in ast.literal_eval(x) if len(item) > 2} if isinstance(x, str) else 
              ({item for item in x if len(item) > 1} if isinstance(x, set) else x))
    return data_set_df

def drop_sets_dataframe_duplicates(dataframe_set_df):
    set_columns = [col for col in dataframe_set_df.columns]
    # print(set_columns)
    for col in set_columns:
        dataframe_set_df[col] = dataframe_set_df[col].apply(
            lambda x: tuple(sorted(x)) if isinstance(x, set) else x)   
    dataframe_set_df = dataframe_set_df.drop_duplicates()
    dataframe_set_df=dataframe_set_df.copy()
    for col in set_columns:
        dataframe_set_df[col] = dataframe_set_df[col].apply(
            lambda x: set(x) if isinstance(x, tuple) else x )
    return dataframe_set_df

def convert_dataframe_to_set(dataframe_df,column,delimiter):
    column_set=column+'_set'
    dataframe_df_split=dataframe_df[column].str.split(delimiter)
    dataframe_df[column_set]=dataframe_df_split.apply(set)
    dataframe_df=discard_empty_sets(dataframe_df,column_set)
    dataframe_df=trim_dataframe_set(dataframe_df,column_set)   
    dataframe_df=discard_set_one_character(dataframe_df,column_set)
    return dataframe_df

def clean_name_set(name_set):
    if not isinstance(name_set, set):
        return set()
    return {
        name.strip()
        for name in name_set
        if isinstance(name, str) and len(name.strip()) > 1}

def clean_name_set_column(name_set):
    if not isinstance(name_set, set):
        return set()
    cleaned_names = set()
    for name in name_set:
        if not isinstance(name, str):
            continue
        # Split name into parts, remove single letters and empty strings
        parts = [p.strip() for p in name.split() if len(p.strip()) > 1]
        if parts:
            cleaned_names.add(' '.join(parts))  # Recombine cleaned parts
    return cleaned_names

def split_set_into_sets(dataframe_set_df,column):
    dataframe_set_df=dataframe_set_df.copy()
    dataframe_set_df[column] = dataframe_set_df[column].apply(
    lambda x: {part.strip() for item in x if isinstance(item, str) for part in item.split() if part.strip()} if isinstance(x, set) else set())
    return dataframe_set_df

def dataframe_contains_match(dataframe_df,search_column,match_column):
    column_out=search_column+'_match'
    dataframe_df[column_out]=dataframe_df.apply(
        lambda row: row[match_column] in row[search_column] if isinstance(row[search_column], str) else False,axis=1)
    return dataframe_df

def  match_2_set_columns(searcher_df,searchee_df,searcher_column,searchee_column):
    # searcher_df_split=searcher_df[searcher_column].str.split(' ')
    # searcher_df=searcher_df_split.apply(set)   
    explode_searchee_df=searchee_df.explode(searchee_column)
    explode_searchee_df_split=explode_searchee_df[searchee_column].str.split(' ')
    explode_searchee_df=explode_searchee_df_split.apply(set)
    searcher_matches=searcher_df[searcher_column]-explode_searchee_df
    searcher_matches
    return searcher_matches

def match_soldier_to_child(soldier_df,child_df,soldier_set_column,children_set_column):
    match_set=match_2_set_columns(soldier_df,child_df,soldier_set_column,children_set_column)  
    child_matches_df=pd.DataFrame(match_set,columns=['child_match'])
    match_index = child_matches_df[child_matches_df['child_match']==set()].index
    
    # Initialized Column to Boolian
    soldier_df['children_match']='False'
    soldier_df.loc[match_index, 'children_match'] ='True'
    return soldier_df

def normalize(s):
    if pd.isna(s):
        return ""
    s = str(s).lower()
    s = re.sub(r'[^a-z\s]', '', s)  # keep only letters and spaces
    s = re.sub(r'\s+', ' ', s)      # collapse multiple spaces
    return s.strip()


def haystack_contains_needle(needle_df,haystack_df):
    mask1 = [
    normalize(parent) in normalize(spouse)
    for parent,spouse in zip(
        needle_df,
        haystack_df,  
    )
    ]
    return mask1

# **** Single_Parent_Child_Match Parent 1 *******
def single_parent_child_match(parent):
    global soldiers_df,abmc_fag_df
    soldiers_df['Parent_1']=soldiers_df['Parents'].str.split('*').str[0].str.strip()
    soldiers_df['Parent_2']=soldiers_df['Parents'].str.split('*').str[1].str.strip()
    soldiers_df1=soldiers_df[['name_index','soldiers_full_name2','soldiers_full_name2_set','Parent_1','Parent_2','birth_year','soldiers_cemetery','entry_state']]
    
    abmc_fag_df1=abmc_fag_df[['full_name','spouses','birth_year','children','children_set','cemetery_name','cemetery_id','prob_jewish'\
                              ,'low_grave_count','memorial_count','birth_state','birth_country']]
    
    matched_parent_spouse_df11=pd.merge(
        soldiers_df1,
        abmc_fag_df1,
        left_on=[parent],
        right_on=['full_name'],    
        how='inner') 
    matched_parent_spouse_df11['Parent']=matched_parent_spouse_df11[parent]
    
    print('matched_parent_spouse_df11-Size:',len(matched_parent_spouse_df11))
    matched_parent_spouse_df11=drop_sets_dataframe_duplicates(matched_parent_spouse_df11)
    matched_parent_spouse_df11.reset_index(drop=True, inplace = True)
    
    # # matched_parent_spouse_df11c=match_soldier_to_child(matched_parent_spouse_df11,matched_parent_spouse_df11,'soldiers_full_name2_set','children_set')
    
    needle_df=matched_parent_spouse_df11['soldiers_full_name2_set']
    haystack_df=matched_parent_spouse_df11['children_set']
    mask=haystack_contains_needle(needle_df,haystack_df)
    matched_parent_spouse_df11['children_match']=pd.DataFrame(mask)
    
    matched_out_df11c=pd.merge(
        soldiers_df1,
        matched_parent_spouse_df11,
        left_on=[ 'name_index'],
        right_on=['name_index'],
        how='inner')
    print('matched_out_df11c-Size:',len(matched_out_df11c))
    matched_out_df11c=drop_sets_dataframe_duplicates(matched_out_df11c)
    matched_out_df11c.reset_index(drop=True, inplace = True)
    print('matched_out_df11c-Size:',len(matched_out_df11c))
    
    matched_out_df11c=drop_sets_dataframe_duplicates(matched_out_df11c)
    matched_out_df11c.reset_index(drop=True, inplace = True)
    print('matched_out_df11c-Size:',len(matched_out_df11c))
    
    reorder_format=['name_index','soldiers_full_name2_x','Parent_1_x','spouses','full_name','children_match','prob_jewish','low_grave_count', 'birth_year_x', 'birth_year_y','birth_state'\
      ,'entry_state_x','birth_country','children','cemetery_name','cemetery_id','memorial_count','soldiers_cemetery_x']
    
    rename_format=['name_index','soldiers_full_name','parent','spouse','full_name','children_match','prob_jewish','low_grave_count', 'birth_year_1', 'birth_year_2','birth_state'\
                    ,'entry_state','birth_country','children','cemetery_name','cemetery_id','memorial_count','soldiers_cemetery']
    
    matched_out_df11c=matched_out_df11c[reorder_format]
    matched_out_df11c.columns=rename_format
    return matched_out_df11c

def  search_dataframe(dataframe,column,search_text):
    dataframe_extract=dataframe[dataframe[column]==search_text]
    return dataframe_extract

In [ ]:
# # OKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOK

# DirMain="C:\\Users\\jrozm\\OneDrive\\Benjamin\\"

# grave_source='ABMC'

# fileInputSoldier="soldiers_df"
# fileInputABMC="abmc_fag_df"
# fileInputSoD="star_david_df"

# fileOutput=grave_source+"_Matched_Soldier_Affiliation"

# fileInSold=DirMain+"Affilation_Analysis\\Inputs\\"+fileInputSoldier+".csv"
# fileInABMC=DirMain+"Affilation_Analysis\\Inputs\\"+fileInputABMC+".csv"
# fileInSoD=DirMain+"Affilation_Analysis\\Inputs\\"+fileInputSoD+".csv"

# soldiers_df=pd.read_csv(fileInSold,encoding='ISO-8859-1', on_bad_lines='skip',low_memory=False)
# print('• soldiers_df * Download Complete *, Length: ',len(soldiers_df))

# abmc_fag_df=pd.read_csv(fileInABMC,encoding='ISO-8859-1', on_bad_lines='skip',low_memory=False)
# print('• abmc_fag_df * Download Complete *, Length: ',len(abmc_fag_df))

# star_david_df=pd.read_csv(fileInSoD)
# print('• Star of David * Download Complete *')

# print(list(soldiers_df),'\n')
# print(list(abmc_fag_df),'\n')
# input('')
# soldiers_df_header=
# ['name_index',
#  'last_name', 
#  'first_name', 
#  'middle_name', 
#  'soldiers_full_name', 
#  'birthDate', 
#  'birth_index', 
#  'deathDate', 
#  'death_index', 
#  'entry_state', 
#  'birthStateName', 
#  'birthCityName', 
#  'birthCountryName', 
#  'soldiers_cemetery', 
#  'links', 'Parents', 
#  'Spouses', 'Siblings', 
#  'Children', 
#  'soldiers_full_name2', 
#  'soldiers_full_name2_set', 
#  'Parents_set', 
#  'Spouses_set', 
#  'Siblings_set', 
#  'Parent_1', 
#  'Parent_2', 
#  'birth_year'] 
# abmc_fag_df_header=
# ['name_index',
#  'name_index_short',
#  'last_name',
#  'first_name',
#  'middle_name',
#  'full_name',
#  'cemetery_name',
#  'cemetery_id',
#  'prob_jewish',
#  'low_grave_count',
#  'memorial_count',
#  'birth_date',
#  'death_date',
#  'birth_index',
#  'death_index',
#  'birth_city',
#  'birth_state',
#  'birth_country',
#  'url',
#  'spouses',
#  'siblings',
#  'children',
#  'full_name_set',
#  'spouses_set',
#  'siblings_set',
#  'children_set',
#  'birth_year'] 

# parent='Parent_1'
# matched_out_df1=single_parent_child_match(parent)
# parent='Parent_2'
# matched_out_df2=single_parent_child_match(parent)
# matched_out_df1=drop_sets_dataframe_duplicates(matched_out_df1)
# matched_out_df1.reset_index(drop=True, inplace = True)
# matched_out_df2=drop_sets_dataframe_duplicates(matched_out_df2)
# matched_out_df2.reset_index(drop=True, inplace = True)


matched_out_df12_inner_df=pd.merge(
    matched_out_df1,
    matched_out_df2,
    left_on=[ 'name_index'],
    right_on=['name_index'],
    how='inner')
# print(list(matched_out_df12_inner_df),'\n • Length: ',len(matched_out_df12_inner_df))






# Condition 1
haystack_df=matched_out_df12_inner_df['spouse_y']
needle_df=matched_out_df12_inner_df['parent_x']
mask1=haystack_contains_needle(needle_df,haystack_df)

haystack_df=matched_out_df12_inner_df['spouse_x']
needle_df=matched_out_df12_inner_df['parent_y']
mask2=haystack_contains_needle(needle_df,haystack_df)

mask12=np.array(mask1) & np.array(mask2)
matched_out_df12_inner_df0=matched_out_df12_inner_df[mask12]
input('')
# matched_out_df12_inner_df=matched_out_df12_inner_df[matched_out_df12_inner_df['children_match_x']==True]
# OKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOKOK

matched_out_df12_left_df=pd.merge(
    matched_out_df1,
    matched_out_df2,
    left_on=[ 'name_index'],
    right_on=['name_index'],
    how='left')

matched_out_df12_left_df=matched_out_df12_left_df[(matched_out_df12_left_df['children_match_x']==True) & \
                         # (matched_out_df12_left_df['prob_jewish_x']>=.5) & \
                        (matched_out_df12_left_df['soldiers_full_name_y'].isna())]

matched_out_df12_right_df=pd.merge(
    matched_out_df1,
    matched_out_df2,
    left_on=[ 'name_index'],
    right_on=['name_index'],
    how='right')

matched_out_df12_right_df=matched_out_df12_right_df[(matched_out_df12_right_df['children_match_y']==True) & \
                         # (matched_out_df12_right_df['prob_jewish_y']>=.5) & \
                        (matched_out_df12_right_df['soldiers_full_name_x'].isna())]

matched_out_df12_inner_df_header=['name_index','soldiers_full_name_x','parent_x','spouse_x','full_name_x','children_match_x','prob_jewish_x','low_grave_count_x','birth_state_x','entry_state_x','birth_country_x','children_x','cemetery_name_x','cemetery_id_x','memorial_count_x','soldiers_cemetery_x','soldiers_full_name_y','parent_y','spouse_y','full_name_y','children_match_y','prob_jewish_y','low_grave_count_y','birth_state_y','entry_state_y','birth_country_y','children_y','cemetery_name_y','cemetery_id_y','memorial_count_y','soldiers_cemetery_y']

x_set=['soldiers_full_name_x','parent_x','spouse_x','full_name_x','children_match_x','prob_jewish_x','low_grave_count_x'\
       ,'birth_state_x','entry_state_x','birth_country_x','children_x','cemetery_name_x','cemetery_id_x'\
       ,'memorial_count_x','soldiers_cemetery_x']

y_set=['soldiers_full_name_y','parent_y','spouse_y','full_name_y','children_match_y','prob_jewish_y','low_grave_count_y'\
       ,'birth_state_y','entry_state_y','birth_country_y','children_y','cemetery_name_y','cemetery_id_y'\
       ,'memorial_count_y','soldiers_cemetery_y']


matched_out_df12_left_df=matched_out_df12_left_df.copy()
matched_out_df12_left_df[y_set]=matched_out_df12_left_df[x_set]
matched_out_df12_left_df['cemetery_name_y']='Left Join'
matched_out_df12_right_df=matched_out_df12_right_df.copy()
matched_out_df12_right_df[x_set]=matched_out_df12_right_df[y_set]
matched_out_df12_right_df['cemetery_name_x']='Right Join'


matched_soldier_affiliation_df_Temp=pd.concat([matched_out_df12_inner_df,matched_out_df12_left_df])
matched_soldier_affiliation_df=pd.concat([matched_soldier_affiliation_df_Temp,matched_out_df12_right_df])
matched_soldier_affiliation_df=drop_sets_dataframe_duplicates(matched_soldier_affiliation_df)
matched_soldier_affiliation_df.reset_index(drop=True, inplace = True)

matched_soldier_affiliation_marker_df=pd.merge(
    matched_soldier_affiliation_df,
    star_david_df,
    left_on=[ 'name_index'],
    right_on=['Name_Index'],
    how='left')


matched_soldier_affiliation_marker_df['Star_of_David'] = matched_soldier_affiliation_marker_df['Name_Index'].fillna(False)
matched_soldier_affiliation_marker_df['Star_of_David']=matched_soldier_affiliation_marker_df['Star_of_David']!=False
matched_soldier_affiliation_marker_df.drop(columns=['Name_Index'], inplace=True)
matched_soldier_affiliation_marker_df.drop_duplicates(inplace=True)

matched_soldier_affiliation_marker_df['confidence']=4

reorder_format=['name_index','soldiers_full_name_x','parent_x','spouse_x','spouse_y','children_match_x','children_match_y','prob_jewish_x','prob_jewish_y'\
         ,'confidence','Star_of_David','cemetery_name_x','cemetery_name_y','cemetery_id_x','cemetery_id_y','soldiers_cemetery_x','children_x','children_y'\
               ,'birth_state_x','birth_state_y','entry_state_x','entry_state_y','birth_country_x','birth_country_y']

rename_format=['name_index','soldiers_full_name','parent','spouse','spouse_match','children_match_1','children_match_2','prob_jewish_1','prob_jewish_2'\
         ,'confidence','Star_of_David','cemetery_name_1','cemetery_name_2','cemetery_id_1','cemetery_id_2','soldiers_cemetery','children_1','children_2'\
         ,'birth_state_1','birth_state_2','entry_state_1','entry_state_2','birth_country_1','birth_country_2']    


matched_soldier_affiliation_marker_df=matched_soldier_affiliation_marker_df[reorder_format]
matched_soldier_affiliation_marker_df.columns=rename_format
matched_soldier_affiliation_marker_df = matched_soldier_affiliation_marker_df.sort_values(by='name_index',ascending=True)
matched_soldier_affiliation_marker_df.drop_duplicates(inplace=True)
matched_soldier_affiliation_marker_df.reset_index(drop=True, inplace = True)

fileOutDf=DirMain+"Affilation_Analysis\\Outputs\\matched_soldier_affiliation_marker_df.csv"
matched_soldier_affiliation_marker_df.to_csv(fileOutDf,mode="w",index=False,header=True)	

print("\n")
print('matched_soldier_affiliation_marker_df - UploadedTo:',fileOutDf,'Number of Rows:',len(matched_soldier_affiliation_marker_df))  

end_time=datetime.datetime.now()
processing_time=end_time-start_time
processing_time_seconds=int(processing_time.total_seconds())
print(f"**** {process_name}   Processing Time: {processing_time_seconds} Sec")    
